# 02 — Construction du signal momentum

**Projet** : Stratégie quantitative momentum sur le STOXX Europe 600  
**Auteur** : ISLEYEN Volkan  
**Date** : Mai 2026

## Objectif

Transformer les prix quotidiens en signal momentum mensuel, puis former les portefeuilles qui seront backtestés dans le notebook 03.

## Définition du signal momentum

Pour chaque titre et à chaque date de rebalancement (fin de mois), on calculele rendement passé sur une fenêtre de **12 mois en sautant le dernier mois** (momentum 12-1) :

$$\text{mom}_{i,t} = \frac{P_{i,t-21}}{P_{i,t-252}} - 1$$

où $t-21$ correspond à environ 1 mois de bourse et $t-252$ à environ 12 mois.

Le "skip" du dernier mois suit la convention de Jegadeesh & Titman (1993) 
et permet d'éviter la contamination du signal par la réversion à court terme.

## Méthodologie

1. Calculer les rendements quotidiens à partir des prix
2. Définir les dates de rebalancement (fin de chaque mois)
3. Calculer le signal momentum 12-1 à chaque rebalancement
4. Classer les titres et former les quintiles
5. Sauvegarder les portefeuilles pour le backtest

## Données en entrée

- `data/processed/prices_clean.csv` — cours des 577 titres

In [1]:
# Librairies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

# Chemins
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW = PROJECT_ROOT / "data" / "raw"
FIGURES = PROJECT_ROOT / "output" / "figures"
TABLES = PROJECT_ROOT / "output" / "tables"

for d in [FIGURES, TABLES]:
    d.mkdir(parents=True, exist_ok=True)

# Chargement des prix nettoyés
prices = pd.read_csv(DATA_PROCESSED / "prices_clean.csv", index_col=0, parse_dates=True)

print(f"Prix chargés : {prices.shape}")
print(f"Période : {prices.index.min().date()} → {prices.index.max().date()}")
print(f"Nombre de titres : {prices.shape[1]}")

Prix chargés : (2586, 577)
Période : 2015-01-01 → 2024-12-30
Nombre de titres : 577


In [3]:
# 1. Rendements quotidiens (simples) à partir des prix
returns = prices.pct_change()
print(f"Rendement quotidients calculés : {returns.shape}")

# 2. Définir les dates de rebalancement = dernier jour de bourse de chaque mois
# On utilise resample sur la fréquence "fin de mois" et on prend le dernier index dispo
month_ends = prices.resample("ME").last().index

# On ne garde que les dates de rebalancement réellement présentes dans nos données
# (le dernier jour de bourse effectif de chaque mois)
rebalance_dates = []
for me in month_ends:
    # Trouver le dernier jour de bourse =< fin de mois théorique
    valid = prices.index[prices.index <= me]
    if len(valid) > 0:
        rebalance_dates.append(valid[-1])

rebalance_dates = pd.DatetimeIndex(rebalance_dates).unique()

print(f"\nNombre de dates de rebalancement : {len(rebalance_dates)}")
print(f"Première : {rebalance_dates[0].date()}")
print(f"Dernière : {rebalance_dates[-1].date()}")
print(f"\nAperçu des premières dates de rebalancement :")
for d in rebalance_dates[:6]:
    print(f"  {d.date()} ({d.day_name()})")

Rendement quotidients calculés : (2586, 577)

Nombre de dates de rebalancement : 120
Première : 2015-01-30
Dernière : 2024-12-30

Aperçu des premières dates de rebalancement :
  2015-01-30 (Friday)
  2015-02-27 (Friday)
  2015-03-31 (Tuesday)
  2015-04-30 (Thursday)
  2015-05-29 (Friday)
  2015-06-30 (Tuesday)


In [6]:
# Calcul du signal momentum 12-1 à chaque date de rebalancement
# mom = rendement entre t-252 jours (≈12 mois) et t-21 jours (≈1 mois)
# On "saute" le dernier mois pour éviter la réversion à court terme

LOOKBACK = 252  # ≈ 12 mois de bourse
SKIP = 21        # ≈ 1 mois de bourse

momentum_signals = {}

for reb_date in rebalance_dates:
     # Position de la date de rebalancement dans l'index des prix
    pos = prices.index.get_loc(reb_date)

    #Il faut au moins LOOKBACK jours historique avant cette date
    if pos < LOOKBACK:
        continue

    # Prix à t-21 (fin de la fenêtre momentum, on saute le dernier mois)
    price_end = prices.iloc[pos-SKIP]

    # Prix à t-252 (début de la fenêtre momentum)
    price_start = prices.iloc[pos - LOOKBACK]

    # Signal momentum = rendement sur la fenêtre [t-252, t-21]
    mom = (price_end / price_start) - 1

    momentum_signals[reb_date] = mom

# Transformer en DataFrame : lignes = dates de rebalancement, colonnes = titres
momentum_df = pd.DataFrame(momentum_signals).T
momentum_df.index.name = "rebalance_date"

print(f"Signal momentum calculé : {momentum_df.shape}")
print(f"Dates de rebalancement exploitables : {len(momentum_df)}")
print(f"  (les premières dates sans 12 mois d'historique sont exclues)")
print(f"\nPremière date avec signal : {momentum_df.index[0].date()}")
print(f"Dernière date avec signal : {momentum_df.index[-1].date()}")

# Aperçu : signal momentum pour quelques titres à la première date
print(f"\nExemple de signaux au {momentum_df.index[0].date()} (5 titres) :")
print(momentum_df.iloc[0, :5].round(4))

Signal momentum calculé : (109, 577)
Dates de rebalancement exploitables : 109
  (les premières dates sans 12 mois d'historique sont exclues)

Première date avec signal : 2015-12-31
Dernière date avec signal : 2024-12-30

Exemple de signaux au 2015-12-31 (5 titres) :
A2A.MI    0.6902
A5G.IR   -0.3671
AAF.L        NaN
AAK.ST    0.5787
AAL.L    -0.6527
Name: 2015-12-31 00:00:00, dtype: float64


In [7]:
# Construction des quintiles à chaque date de rebalancement
# Q5 = top quintile (gagnants, ceux qu'on achète)
# Q1 = bottom quintile (perdants)

N_QUINTILES = 5

# Pour chaque date, on attribue à chaque titre son quintile (1 à 5)
quintile_assignments = {}

for reb_date in momentum_df.index:
    signals = momentum_df.loc[reb_date].dropna()  # on enlève les titres sans signal
    
    # Il faut un minimum de titres pour faire des quintiles cohérents
    if len(signals) < N_QUINTILES * 5:  # au moins 25 titres
        continue
    
    # qcut découpe en quintiles de taille égale, labels 1 à 5
    # 5 = meilleur momentum (top quintile), 1 = pire
    quintiles = pd.qcut(signals, N_QUINTILES, labels=range(1, N_QUINTILES + 1))
    quintile_assignments[reb_date] = quintiles

# Compter combien de titres dans chaque quintile à chaque date
print("Vérification de la composition des quintiles\n")

# Exemple sur la première date
first_date = list(quintile_assignments.keys())[0]
first_quintiles = quintile_assignments[first_date]

print(f"Date : {first_date.date()}")
print(f"Nombre de titres classés : {len(first_quintiles)}")
print(f"\nRépartition par quintile :")
print(first_quintiles.value_counts().sort_index())

# Les titres du top quintile (Q5) à cette date
top_quintile_titres = first_quintiles[first_quintiles == 5].index.tolist()
print(f"\nTop quintile (Q5) au {first_date.date()} : {len(top_quintile_titres)} titres")
print(f"Premiers titres gagnants : {top_quintile_titres[:10]}")

Vérification de la composition des quintiles

Date : 2015-12-31
Nombre de titres classés : 515

Répartition par quintile :
2015-12-31 00:00:00
1    103
2    103
3    103
4    103
5    103
Name: count, dtype: int64

Top quintile (Q5) au 2015-12-31 : 103 titres
Premiers titres gagnants : ['A2A.MI', 'AAK.ST', 'ADS.DE', 'AGS.BR', 'AIR.PA', 'AKRBP.OL', 'AMBU-B.CO', 'ANA.MC', 'AXFO.ST', 'AZA.ST']


In [8]:
# Construction de la matrice de poids du portefeuille momentum
# Stratégie : long-only, equal-weight sur le top quintile (Q5)

# DataFrame de poids : index = toutes les dates de prix, colonnes = titres
weights = pd.DataFrame(0.0, index=prices.index, columns=prices.columns)

# Pour chaque date de rebalancement, on définit les poids jusqu'au rebalancement suivant
reb_dates_with_signal = list(quintile_assignments.keys())

for i, reb_date in enumerate(reb_dates_with_signal):
    quintiles = quintile_assignments[reb_date]
    
    # Titres du top quintile (Q5 = gagnants)
    winners = quintiles[quintiles == 5].index.tolist()
    
    # Poids equal-weight : 1/N réparti sur les gagnants
    w = 1.0 / len(winners)
    
    # Période d'application : de la date de rebalancement (exclue) 
    # jusqu'au prochain rebalancement (inclus)
    start_apply = reb_date
    if i + 1 < len(reb_dates_with_signal):
        end_apply = reb_dates_with_signal[i + 1]
    else:
        end_apply = prices.index[-1]
    
    # Masque des dates concernées (après le rebalancement, jusqu'au suivant)
    mask = (prices.index > start_apply) & (prices.index <= end_apply)
    
    # Affecter les poids aux titres gagnants sur cette période
    weights.loc[mask, winners] = w

print(f"Matrice de poids construite : {weights.shape}")

# Vérification : la somme des poids par jour doit être ≈ 1 (ou 0 avant le 1er rebalancement)
poids_sum = weights.sum(axis=1)
print(f"\nSomme des poids par jour :")
print(f"  Jours à 0 (avant 1er rebalancement) : {(poids_sum == 0).sum()}")
print(f"  Jours à ≈1 (portefeuille investi)   : {(np.abs(poids_sum - 1) < 0.001).sum()}")

# Aperçu : nombre de titres détenus dans le temps
n_holdings = (weights > 0).sum(axis=1)
print(f"\nNombre de titres détenus :")
print(f"  Min : {n_holdings[n_holdings > 0].min()}")
print(f"  Max : {n_holdings.max()}")
print(f"  Moyenne (jours investis) : {n_holdings[n_holdings > 0].mean():.0f}")

Matrice de poids construite : (2586, 577)

Somme des poids par jour :
  Jours à 0 (avant 1er rebalancement) : 261
  Jours à ≈1 (portefeuille investi)   : 2325

Nombre de titres détenus :
  Min : 49
  Max : 116
  Moyenne (jours investis) : 108


In [9]:
# Sauvegarde des éléments nécessaires au backtest

# 1. Matrice de poids
weights.to_csv(DATA_PROCESSED / "weights_momentum.csv")
print(f"Poids sauvegardés : {weights.shape}")

# 2. Signal momentum (pour analyses ultérieures éventuelles)
momentum_df.to_csv(DATA_PROCESSED / "momentum_signals.csv")
print(f"Signaux momentum sauvegardés : {momentum_df.shape}")

# 3. Rendements quotidiens (pour le calcul de performance)
returns.to_csv(DATA_PROCESSED / "returns_daily.csv")
print(f"Rendements quotidiens sauvegardés : {returns.shape}")

# 4. Dates de rebalancement avec signal (pour le calcul du turnover)
pd.Series(reb_dates_with_signal).to_csv(
    DATA_PROCESSED / "rebalance_dates.csv", index=False, header=["rebalance_date"]
)
print(f"Dates de rebalancement sauvegardées : {len(reb_dates_with_signal)}")

print("\nTous les fichiers nécessaires au backtest sont prêts.")

Poids sauvegardés : (2586, 577)
Signaux momentum sauvegardés : (109, 577)
Rendements quotidiens sauvegardés : (2586, 577)
Dates de rebalancement sauvegardées : 108

Tous les fichiers nécessaires au backtest sont prêts.


## Récapitulatif du notebook 02

### Signal construit

Le signal momentum 12-1 (rendement sur 12 mois en sautant le dernier mois) 
a été calculé pour les 577 titres à chaque date de rebalancement mensuelle, 
suivant la convention de Jegadeesh & Titman (1993).

### Portefeuille formé

- **Stratégie** : long-only, equal-weight sur le top quintile (Q5)
- **109 dates de rebalancement** exploitables (fin 2015 à fin 2024)
- **~108 titres détenus** en moyenne (top 20% des titres par momentum)
- **Rebalancement mensuel**

### Choix méthodologiques clés

**Skip d'un mois** : le signal exclut le dernier mois (fenêtre [-252, -21] jours) 
pour éviter la contamination par la réversion à court terme.

**Prévention du look-ahead bias** : les poids calculés à une date de 
rebalancement ne sont appliqués qu'à partir du jour de bourse suivant. La 
décision d'investissement n'utilise donc que l'information disponible au 
moment de la décision.

### Fichiers produits

- `data/processed/weights_momentum.csv` — matrice de poids quotidiens
- `data/processed/momentum_signals.csv` — signaux momentum par rebalancement
- `data/processed/returns_daily.csv` — rendements quotidiens des titres
- `data/processed/rebalance_dates.csv` — dates de rebalancement

### Prochaine étape

Notebook 03 : backtest de la stratégie (calcul de la performance, application 
des coûts de transaction) et comparaison au benchmark STOXX 600.